# Challenge 1: Comparing MLP Frameworks

This lab implements the Challenge 1 comparison using the MNIST handwritten-digit dataset. Each framework trains a comparable two-hidden-layer MLP for the same number of epochs, then reports **training time** and **test accuracy**. Timings are hardware-dependent, so the benchmark should be rerun in the target environment.


In [1]:
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras

SEED = 8010
EPOCHS = 3
BATCH_SIZE = 128
HIDDEN_UNITS = (128, 64)

np.random.seed(SEED)
tf.random.set_seed(SEED)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") / 255.0).reshape(-1, 784)
x_test = (x_test.astype("float32") / 255.0).reshape(-1, 784)

print(f"Training examples: {len(x_train):,}")
print(f"Test examples: {len(x_test):,}")


Training examples: 60,000
Test examples: 10,000


## Keras

Keras provides the shortest implementation. Its `fit` method manages batching, gradient computation, and metric tracking.


In [2]:
def build_keras_model():
    return keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(HIDDEN_UNITS[0], activation="relu"),
        keras.layers.Dense(HIDDEN_UNITS[1], activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])

keras_model = build_keras_model()
keras_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

start = time.perf_counter()
keras_model.fit(x_train, y_train, batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=0)
keras_training_seconds = time.perf_counter() - start
_, keras_accuracy = keras_model.evaluate(x_test, y_test, verbose=0)

print(f"Keras training time: {keras_training_seconds:.2f} seconds")
print(f"Keras test accuracy: {keras_accuracy:.4f}")


Keras training time: 8.92 seconds
Keras test accuracy: 0.9714


## PyTorch

PyTorch exposes the training loop explicitly. The implementation uses the same normalized NumPy arrays as Keras and TensorFlow, converted to tensors without relying on `torchvision`.


In [4]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as error:
    raise ImportError(
        "PyTorch is required for this comparison. Install the repository requirements before running this section."
    ) from error

torch.manual_seed(SEED)

class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(784, HIDDEN_UNITS[0]),
            nn.ReLU(),
            nn.Linear(HIDDEN_UNITS[0], HIDDEN_UNITS[1]),
            nn.ReLU(),
            nn.Linear(HIDDEN_UNITS[1], 10),
        )

    def forward(self, inputs):
        return self.network(inputs)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train).long()),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test).long()),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

torch_model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(torch_model.parameters())

start = time.perf_counter()
torch_model.train()
for _ in range(EPOCHS):
    for features, labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(torch_model(features), labels)
        loss.backward()
        optimizer.step()
torch_training_seconds = time.perf_counter() - start

torch_model.eval()
correct = 0
with torch.no_grad():
    for features, labels in test_loader:
        correct += (torch_model(features).argmax(dim=1) == labels).sum().item()
torch_accuracy = correct / len(test_loader.dataset)

print(f"PyTorch training time: {torch_training_seconds:.2f} seconds")
print(f"PyTorch test accuracy: {torch_accuracy:.4f}")


PyTorch training time: 14.34 seconds
PyTorch test accuracy: 0.9658


## TensorFlow Core API

TensorFlow Core makes the forward pass, gradient calculation, and optimizer update explicit with `tf.GradientTape`. The first traced call can add one-time graph-construction overhead to the measured time.


In [5]:
tf_train = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(
    len(x_train), seed=SEED, reshuffle_each_iteration=True
).batch(BATCH_SIZE)
tf_test = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE)

class TensorFlowMLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.hidden1 = tf.keras.layers.Dense(HIDDEN_UNITS[0], activation="relu")
        self.hidden2 = tf.keras.layers.Dense(HIDDEN_UNITS[1], activation="relu")
        self.output_layer = tf.keras.layers.Dense(10)

    def call(self, inputs, training=False):
        del training
        return self.output_layer(self.hidden2(self.hidden1(inputs)))

tf_model = TensorFlowMLP()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
tf_optimizer = tf.keras.optimizers.Adam()

@tf.function
def train_step(features, labels):
    with tf.GradientTape() as tape:
        logits = tf_model(features, training=True)
        loss = loss_fn(labels, logits)
    gradients = tape.gradient(loss, tf_model.trainable_variables)
    tf_optimizer.apply_gradients(zip(gradients, tf_model.trainable_variables))
    return loss

start = time.perf_counter()
for _ in range(EPOCHS):
    for features, labels in tf_train:
        train_step(features, labels)
tf_training_seconds = time.perf_counter() - start

metric = tf.keras.metrics.SparseCategoricalAccuracy()
for features, labels in tf_test:
    metric.update_state(labels, tf_model(features, training=False))
tf_accuracy = float(metric.result())

print(f"TensorFlow Core training time: {tf_training_seconds:.2f} seconds")
print(f"TensorFlow Core test accuracy: {tf_accuracy:.4f}")


TensorFlow Core training time: 7.24 seconds
TensorFlow Core test accuracy: 0.9731


## Comparison

The table below is generated from the measurements above, so it remains accurate when the notebook is rerun on different hardware.


In [6]:
results = [
    ("Keras", keras_training_seconds, float(keras_accuracy)),
    ("PyTorch", torch_training_seconds, torch_accuracy),
    ("TensorFlow Core", tf_training_seconds, tf_accuracy),
]

print("| Framework | Training time (s) | Test accuracy |")
print("|---|---:|---:|")
for framework, seconds, accuracy in results:
    print(f"| {framework} | {seconds:.2f} | {accuracy:.4f} |")

fastest = min(results, key=lambda result: result[1])
most_accurate = max(results, key=lambda result: result[2])
print(f"\nFastest in this run: {fastest[0]}")
print(f"Highest accuracy in this run: {most_accurate[0]}")


| Framework | Training time (s) | Test accuracy |
|---|---:|---:|
| Keras | 8.92 | 0.9714 |
| PyTorch | 14.34 | 0.9658 |
| TensorFlow Core | 7.24 | 0.9731 |

Fastest in this run: TensorFlow Core
Highest accuracy in this run: TensorFlow Core


### Interpretation

Keras is the most concise choice for a standard MNIST classifier and is therefore my preferred implementation for this task. PyTorch is preferable when the training loop needs to be customized or debugged interactively. TensorFlow Core provides the most direct control over gradient computation and can benefit from graph execution, but requires the most code. Accuracy should be compared alongside timing: a small speed difference is less important if one implementation is less accurate or harder to maintain.
